In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report


df = pd.read_csv('./data/raw/Dataset.csv')
print("Dataset Shape:", df.shape)
df.head()

df = df.dropna(subset=['type', 'duration', 'rating', 'listed_in', 'country'])


features = ['release_year', 'rating', 'duration', 'listed_in', 'country']
target = 'type'


Dataset Shape: (8790, 10)


In [ ]:

le_type = LabelEncoder()
y = le_type.fit_transform(df[target]) 


X_duration = df['duration'].str.extract(r'(\d+)').astype(float)
X_duration.columns = ['duration_num']


X_year = df[['release_year']].reset_index(drop=True)


X_categorical = pd.get_dummies(df[['rating', 'country']], drop_first=True).reset_index(drop=True)


cv = CountVectorizer(tokenizer=lambda x: [i.strip() for i in x.split(',')], lowercase=False)
X_genres = pd.DataFrame(cv.fit_transform(df['listed_in']).toarray(), columns=cv.get_feature_names_out())


X = pd.concat([X_duration, X_year, X_categorical, X_genres], axis=1)
X.columns = X.columns.astype(str) # Ensure all column names are strings


/Users/adityakamble/miniconda3/envs/contentTypePrediction/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# Train models
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model


In [ ]:
results = []

print("--- CLASSIFICATION REPORTS ---\n")
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    results.append({"Model": name, "Accuracy": acc})
    
    print(f"[{name}]")
    print(classification_report(y_test, y_pred, target_names=le_type.classes_, zero_division=0))
    print("-" * 40)


comparison_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)

print("\n--- STEP 5: MODEL COMPARISON SUMMARY ---")
print(comparison_df.to_string(index=False))


--- CLASSIFICATION REPORTS ---

[Logistic Regression]
              precision    recall  f1-score   support

       Movie       1.00      1.00      1.00      1238
     TV Show       0.99      1.00      1.00       520

    accuracy                           1.00      1758
   macro avg       1.00      1.00      1.00      1758
weighted avg       1.00      1.00      1.00      1758

----------------------------------------
[Decision Tree]
              precision    recall  f1-score   support

       Movie       1.00      1.00      1.00      1238
     TV Show       0.99      1.00      1.00       520

    accuracy                           1.00      1758
   macro avg       1.00      1.00      1.00      1758
weighted avg       1.00      1.00      1.00      1758

----------------------------------------
[Random Forest]
              precision    recall  f1-score   support

       Movie       1.00      1.00      1.00      1238
     TV Show       0.99      1.00      1.00       520

    accuracy  

In [ ]:
import joblib


best_model = trained_models["Random Forest"]


joblib.dump(best_model, 'netflix_rf_model.pkl')
joblib.dump(le_type, 'netflix_label_encoder.pkl')
joblib.dump(X_train.columns, 'netflix_model_columns.pkl')

print("Model and dependencies saved successfully!")


Model and dependencies saved successfully!


In [ ]:

loaded_model = joblib.load('netflix_rf_model.pkl')
loaded_le = joblib.load('netflix_label_encoder.pkl')
model_columns = joblib.load('netflix_model_columns.pkl')

def predict_netflix_type(duration_num, release_year, rating, country, genres):

    input_data = pd.DataFrame(0, index=[0], columns=model_columns)
    

    input_data['duration_num'] = duration_num
    input_data['release_year'] = release_year
    
    rating_col = f'rating_{rating}'
    if rating_col in model_columns:
        input_data[rating_col] = 1
        

    country_col = f'country_{country}'
    if country_col in model_columns:
        input_data[country_col] = 1
        

    for genre in genres:
        if genre in model_columns:
            input_data[genre] = 1
            
    # 6. Make prediction
    prediction_num = loaded_model.predict(input_data)
    prediction_label = loaded_le.inverse_transform(prediction_num)
    
    return prediction_label[0]


test_1 = predict_netflix_type(
    duration_num=3, 
    release_year=2022, 
    rating='TV-MA', 
    country='United Kingdom', 
    genres=['Crime TV Shows', 'TV Dramas']
)
print(f"Test 1 Prediction: {test_1}")  # Expected: TV Show


test_2 = predict_netflix_type(
    duration_num=110, 
    release_year=2023, 
    rating='PG-13', 
    country='United States', 
    genres=['Action & Adventure']
)
print(f"Test 2 Prediction: {test_2}")  # Expected: Movie


Test 1 Prediction: TV Show
Test 2 Prediction: Movie
